# Match Simbad Targets with LSSTCam Sources Catalogue

For each target star (identified by `simbad_id`, `ra_deg`, `dec_deg`) read from
the input CSV, this notebook:

1. Locates the Butler tract/patch that covers the target coordinates.
2. Loads the corresponding `source` (or equivalent catalogue dataset
   auto-discovered from the registry).
3. Performs a nearest-neighbour sky cross-match using
   `astropy.coordinates.SkyCoord.match_to_catalog_sky`.
4. Retains matches within a configurable search radius.
5. Saves the merged table (targets + matched LSST object columns) to CSV/Parquet.


- refer to : https://github.com/lsst/tutorial-notebooks/blob/main/DP1/200_Data_Products/201_Catalogs/201_2_Source_table.ipynb

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-06-26
- **Last update:** 2026-06-26


## Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from astropy.coordinates import SkyCoord
import astropy.units as u
from astropy.time import Time

from lsst.daf.butler import Butler, Timespan
import lsst.geom as geom
from lsst.geom import SpherePoint, degrees

## Helper utilities

In [ ]:
def dataset_type_exists(butler, name):
    """Return True if *name* is a registered dataset type in *butler*."""
    try:
        butler.registry.getDatasetType(name)
        return True
    except KeyError:
        return False

## Configuration

**Edit only this cell** to point to the right Butler repository, collections,
input CSV, search radius, and output band.


In [ ]:
# ── Output directories ───────────────────────────────────────────────────
NB_TAG = "MATCHSRC_03"
DIR_DATA_IN = "data_DEEPCCUTOUTS_01_in"  # reuse the same input directory as NB01
DIR_DATA_OUT = f"data_{NB_TAG}_out"
DIR_FIGS = f"figs_{NB_TAG}"
os.makedirs(DIR_DATA_OUT, exist_ok=True)
os.makedirs(DIR_FIGS, exist_ok=True)
print(f"Data input : {os.path.abspath(DIR_DATA_IN)}")
print(f"Data output: {os.path.abspath(DIR_DATA_OUT)}")
print(f"Figs       : {os.path.abspath(DIR_FIGS)}")

# ── Matplotlib style ────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 120,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "font.size": 9,
    }
)


def savefig(name: str) -> None:
    """Save the current figure to PDF and PNG inside DIR_FIGS."""
    for ext in ("pdf", "png"):
        plt.savefig(os.path.join(DIR_FIGS, f"{name}.{ext}"), bbox_inches="tight")
    print(f"  -> saved {name}.{{pdf,png}}")


print("Configuration done.")

In [ ]:
# ── Butler ────────────────────────────────────────────────────────────────
repo = "dp2_prep"

collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

instrument = "LSSTCam"
skymapName = "lsst_cells_v2"

BANDSEL = "r"

# ── Input targets ─────────────────────────────────────────────────────
target_file = "summary_visit_counts_per_star_V17-21_r2.0deg.csv"

# ── Cross-match search radius ─────────────────────────────────────────────
MATCH_RADIUS_ARCSEC = 1.0  # maximum separation for a valid match [arcsec]

DATE_START = "2025-04-01T00:00:00"
DATE_STOP = "2026-07-01T00:00:00"
time_start = Time(DATE_START, format="isot", scale="utc")
time_stop = Time(DATE_STOP, format="isot", scale="utc")
MJD_START = time_start.mjd
MJD_STOP = time_stop.mjd
DELTAMJD_DAYS = MJD_STOP - MJD_START
print(f"MJD ::: start = {MJD_START} , stop = {MJD_STOP} , delta t = {DELTAMJD_DAYS} days")

SRC_COLUMNS = [
    "coord_ra",
    "coord_dec",
    "parentSourceId",
    "x",
    "y",
    "xErr",
    "yErr",
    "ra",
    "dec",
    "raErr",
    "decErr",
    "calibFlux",
    "calibFluxErr",
    # "ap03Flux",
    # "ap03FluxErr",
    # "ap03Flux_flag",
    # "ap06Flux",
    # "ap06FluxErr",
    # "ap06Flux_flag",
    "ap09Flux",
    "ap09FluxErr",
    "ap09Flux_flag",
    "ap12Flux",
    "ap12FluxErr",
    "ap12Flux_flag",
    "ap17Flux",
    "ap17FluxErr",
    "ap17Flux_flag",
    "ap25Flux",
    "ap25FluxErr",
    "ap25Flux_flag",
    "ap35Flux",
    "ap35FluxErr",
    "ap35Flux_flag",
    # "ap50Flux",
    # "ap50FluxErr",
    # "ap50Flux_flag",
    # "ap70Flux",
    # "ap70FluxErr",
    # "ap70Flux_flag",
    "sky",
    "skyErr",
    "psfFlux",
    "psfFluxErr",
    #'ixx','iyy','ixy','ixxPSF','iyyPSF','ixyPSF','ixxDebiasedPSF','iyyDebiasedPSF','ixyDebiasedPSF',
    #'gaussianFlux','gaussianFluxErr',
    "extendedness",
    "sizeExtendedness",
    #'blendedness_abs','blendedness_flag','blendedness_flag_noCentroid','blendedness_flag_noShape',
    "apFlux_12_0_flag",
    # "apFlux_12_0_flag_apertureTruncated",
    # "apFlux_12_0_flag_sincCoeffsTruncated",
    "apFlux_12_0_instFlux",
    "apFlux_12_0_instFluxErr",
    "apFlux_17_0_flag",
    "apFlux_17_0_instFlux",
    "apFlux_17_0_instFluxErr",
    "apFlux_35_0_flag",
    "apFlux_35_0_instFlux",
    "apFlux_35_0_instFluxErr",
    # "apFlux_50_0_flag",
    # "apFlux_50_0_instFlux",
    # "apFlux_50_0_instFluxErr",
    #'normCompTophatFlux_flag','normCompTophatFlux_instFlux','normCompTophatFlux_instFluxErr',
    "extendedness_flag",
    "sizeExtendedness_flag",
    #'footprintArea_value','invalidPsfFlag','jacobian_flag','jacobian_value',
    "localBackground_instFlux",
    "localBackground_instFluxErr",
    "localBackground_flag",
    # "localBackground_flag_noGoodPixels",
    # "localBackground_flag_noPsf",
    #'pixelFlags_bad','pixelFlags_cr','pixelFlags_crCenter','pixelFlags_edge','pixelFlags_interpolated','pixelFlags_interpolatedCenter','pixelFlags_nodata','pixelFlags_offimage','pixelFlags_saturated','pixelFlags_saturatedCenter','pixelFlags_suspect','pixelFlags_suspectCenter',
    #'psfFlux_apCorr','psfFlux_apCorrErr',
    #'psfFlux_area','psfFlux_flag','psfFlux_flag_apCorr','psfFlux_flag_edge','psfFlux_flag_noGoodPixels',
    #'gaussianFlux_flag','centroid_flag','centroid_flag_almostNoSecondDerivative','centroid_flag_badError','centroid_flag_edge','centroid_flag_noSecondDerivative','centroid_flag_notAtMaximum','centroid_flag_resetToPeak',
    #'variance_flag','variance_flag_emptyFootprint','variance_value',
    #'calib_astrometry_used','calib_photometry_reserved','calib_photometry_used','calib_psf_candidate','calib_psf_reserved','calib_psf_used',
    #'deblend_deblendedAsPsf','deblend_hasStrayFlux','deblend_masked','deblend_nChild','deblend_parentTooBig','deblend_patchedTemplate','deblend_rampedTemplate','deblend_skipped','deblend_tooManyPeaks',
    #'hsmPsfMoments_flag','hsmPsfMoments_flag_no_pixels','hsmPsfMoments_flag_not_contained','hsmPsfMoments_flag_parent_source',
    #'iDebiasedPSF_flag','iDebiasedPSF_flag_no_pixels','iDebiasedPSF_flag_not_contained','iDebiasedPSF_flag_parent_source','iDebiasedPSF_flag_galsim','iDebiasedPSF_flag_edge',
    #'hsmShapeRegauss_flag','hsmShapeRegauss_flag_galsim','hsmShapeRegauss_flag_no_pixels','hsmShapeRegauss_flag_not_contained','hsmShapeRegauss_flag_parent_source',
    "sky_source",
    "visit",
    "detector",
    "band",
    "physical_filter",
    "sourceId",
]

print("Butler configuration done.")

## Load targets

In [ ]:
df_targets = pd.read_csv(os.path.join(DIR_DATA_IN, target_file))
print(f"Loaded {len(df_targets)} targets from {target_file}")
display(df_targets.head())

## Initialise the Butler

In [ ]:
butler = Butler(repo, collections=collection)
registry = butler.registry
skymap = butler.get("skyMap", skymap=skymapName, collections=collection)
print(f"Butler initialised | repo: {repo}")

## Auto-discover the object-table dataset type

The catalogue dataset name changed across pipeline versions:

| Pipeline era | Dataset type name          |
|:-------------|:---------------------------|
| Gen2 / HSC   | `deepCoadd_obj`            |
| DP1          | `objectTable_tract`        |
| DP2+         | `object_table_tract`       |

We probe the registry so the notebook is collection-agnostic.


In [ ]:
# Prioritised list of candidate object-table dataset type names
SRC_TABLE_CANDIDATES = [
    "source",
    "sourceTable," "sourceTable_visit",  # visit-level source table (fallback)
]

# List all source-table-related types actually in the registry
all_src_types = [
    d.name for d in registry.queryDatasetTypes() if "source" in d.name.lower() or "table" in d.name.lower()
]
print("src-table-related dataset types in registry:")
# for t in sorted(all_src_types):
#    print(f"  {t}")

# Pick the first candidate that is registered
SRC_DATASET = None
for name in SRC_TABLE_CANDIDATES:
    if dataset_type_exists(butler, name):
        SRC_DATASET = name
        print(f"\n✔ Selected src-table dataset type: '{SRC_DATASET}'")
        break

if SRC_DATASET is None:
    raise RuntimeError(
        "No recognised source-table dataset type found in this Butler collection. "
        f"Candidate types seen: {all_src_types}"
    )

## Inspect the schema of the source table (probe on first target)

We load a single source table from a representative visit to check which
columns are available before the main loop.

In [ ]:
# Use the coordinates of the first target to locate a representative tract
first_row = df_targets.iloc[0]

target_name = first_row["simbad_id"]
target_field = first_row["field"]
target_ra = first_row["ra_deg"]
target_dec = first_row["dec_deg"]

probe_point = SpherePoint(target_ra, target_dec, degrees)
probe_tract_info = skymap.findTract(probe_point)
probe_patch_info = probe_tract_info.findPatch(probe_point)
probe_tract_id = probe_tract_info.getId()
probe_patch_id = probe_patch_info.getSequentialIndex()
probe_dataId = {"tract": probe_tract_id, "patch": probe_patch_id, "skymap": skymapName}

# Build timespan
time1 = Time(MJD_START, format="mjd", scale="tai")
time2 = Time(MJD_STOP, format="mjd", scale="tai")
timespan = Timespan(time1, time2)

# Query source refs for the probe star — materialise into a list
probe_refs = list(
    butler.query_datasets(
        "source",
        where=(
            "visit.timespan OVERLAPS :timespan AND " "visit_detector_region.region OVERLAPS POINT(:ra, :dec)"
        ),
        bind={"timespan": timespan, "ra": target_ra, "dec": target_dec},
    )
)

# Number of refs == number of (visit, detector) pairs covering this position
print(f"Number of source refs (visits x detectors): {len(probe_refs)}")
if probe_refs:
    print("Example dataId:", probe_refs[0].dataId)

# Load the first source table to probe the schema
df_probe = butler.get(probe_refs[0], parameters={"columns": SRC_COLUMNS})
if not isinstance(df_probe, pd.DataFrame):
    df_probe = df_probe.to_pandas()
print(f"Probe source table: {len(df_probe)} rows, {len(df_probe.columns)} columns")

In [ ]:
print("Number of source rows in probe table:", len(df_probe))
print(df_probe.columns.tolist())

In [ ]:
# Identify the RA/Dec and sourceId column names from the schema


def find_col(df, candidates):
    """Return the first column name from *candidates* that exists in *df*."""
    for c in candidates:
        if c in df.columns:
            return c
    raise KeyError(f"None of {candidates} found in columns: {list(df.columns[:20])}")


RA_COL = find_col(df_probe, ["coord_ra", "ra", "RA", "ra_deg"])
DEC_COL = find_col(df_probe, ["coord_dec", "dec", "DEC", "dec_deg"])
ID_COL = find_col(df_probe, ["sourceId", "objectId", "object_id", "id"])

print(f"RA  column : {RA_COL}")
print(f"Dec column : {DEC_COL}")
print(f"ID  column : {ID_COL}")

# Restrict SRC_COLUMNS to those actually present in this schema
SRC_COLUMNS_AVAIL = [c for c in SRC_COLUMNS if c in df_probe.columns]
print(f"Requested columns available: {len(SRC_COLUMNS_AVAIL)}/{len(SRC_COLUMNS)}")

## Source-table cache helper

This notebook retrieves *per-visit source* measurements (not coadd objects).
The cross-match strategy for the light-curve extraction is:

- For each `(visit, detector)` ref that overlaps the target position, load the source table.
- Find the nearest source to the target with `SkyCoord`.
- Record `psfFlux` + all photometric metadata.

The light curve is assembled from all matched single-visit rows.


In [ ]:
# Simple in-memory cache: (visit,band,instrument) → pd.DataFrame
src_cache: dict = {}


def get_source_table(ref) -> pd.DataFrame | None:
    """
    Load (and cache) the source table for a single Butler dataset ref.
    """
    key = (ref.dataId["visit"], ref.dataId["band"], ref.dataId["instrument"])

    if key in src_cache:
        return src_cache[key]
    try:
        df = butler.get(ref, parameters={"columns": SRC_COLUMNS})
        if not isinstance(df, pd.DataFrame):
            df = df.to_pandas()
        # Keep only the columns we care about (those available in the schema)
        keep = [c for c in SRC_COLUMNS_AVAIL if c in df.columns]
        df = df[keep].copy()
        src_cache[key] = df
        return df
    except Exception as exc:
        print(f"  WARNING: could not load ref {key}: {exc}")
        return None


print("Source-table cache helper ready.")

## Light-curve extraction loop

For every Simbad target we:
1. Query all `source` refs whose detector footprint overlaps the target sky position inside the chosen time window.
2. For each ref, load the source table and find the nearest source to the target with `SkyCoord.match_to_catalog_sky`.
3. Keep the match if separation ≤ `MATCH_RADIUS_ARCSEC`.
4. Collect visit metadata (band, day_obs, visit id) + photometric columns.

> **Note on time axis:** The precise MJD per visit will be added in a later
> notebook by joining with the `visitTable` / `consDb` visit table.
> Here we only record `visit` and `day_obs` as integer identifiers.


In [ ]:
# ─── Main light-curve extraction loop ─────────────────────────────────────────────
# lc_rows      : one row per (target, visit, detector) match
# match_summary: one row per target (cross-match statistics)

lc_rows: list[dict] = []
match_summary: list[dict] = []

# loop on simbad target
for idx, target in df_targets.iterrows():
    if idx >= 1:
        continue

    simbad_id = target["simbad_id"]
    ra_t = target["ra_deg"]
    dec_t = target["dec_deg"]
    tgt_sky = SkyCoord(ra=ra_t * u.deg, dec=dec_t * u.deg)

    print(f"[{idx:3d}] {simbad_id}  ra={ra_t:.5f}  dec={dec_t:+.5f}")

    # ── 1.  Find all source refs overlapping this position ─────────────────────
    try:
        refs = list(
            butler.query_datasets(
                "source",
                where=(
                    "visit.timespan OVERLAPS :timespan AND "
                    "visit_detector_region.region OVERLAPS POINT(:ra, :dec)"
                ),
                bind={"timespan": timespan, "ra": ra_t, "dec": dec_t},
            )
        )
    except Exception as exc:
        print(f"  ERROR querying refs: {exc}")
        match_summary.append(
            {
                "simbad_id": simbad_id,
                "ra_deg": ra_t,
                "dec_deg": dec_t,
                "n_refs": 0,
                "n_matched": 0,
                "n_failed": 0,
                "status": "query_error",
            }
        )
        continue

    print(f"  → {len(refs)} refs (visit×detector pairs)")

    n_matched = 0
    n_failed = 0

    # loop on refs
    for count, ref in enumerate(refs):
        if count > 100:
            continue

        did = ref.dataId
        visit = did["visit"]
        band = did.get("band", "?")
        day_obs = did.get("day_obs", -1)
        instrument = did.get("instrument", -1)
        physical_filter = did.get("physical_filter", "?")

        if count % 20 == 0:
            print(f" {count} :: {day_obs}, {visit}, {band} ")

        # ── 2.  Load source table ──────────────────────────────────────────
        df_src = get_source_table(ref)
        if df_src is None or len(df_src) == 0:
            n_failed += 1
            continue

        # ── 3.  Build SkyCoord catalogue ───────────────────────────────────
        ra_arr = df_src[RA_COL].values
        dec_arr = df_src[DEC_COL].values

        # Heuristic: radians if max < 2π + ε
        unit_sky = u.rad if ra_arr.max() <= 2 * np.pi + 0.1 else u.deg
        cat_sky = SkyCoord(ra=ra_arr * unit_sky, dec=dec_arr * unit_sky)

        # ── 4.  Nearest-neighbour match ────────────────────────────────────
        best_i, sep2d, _ = tgt_sky.match_to_catalog_sky(cat_sky)
        sep_arcsec = sep2d.to(u.arcsec).value

        if sep_arcsec > MATCH_RADIUS_ARCSEC:
            # No source close enough in this exposure
            continue

        n_matched += 1
        matched_src = df_src.iloc[best_i].copy()
        del df_src

        # Convert matched RA/Dec to degrees for output
        m_ra = matched_src[RA_COL]
        m_dec = matched_src[DEC_COL]
        if unit_sky == u.rad:
            m_ra = np.degrees(m_ra)
            m_dec = np.degrees(m_dec)

        # ── 5.  Build output row ──────────────────────────────────────────
        row = {
            # Target identification
            "simbad_id": simbad_id,
            "target_ra": ra_t,
            "target_dec": dec_t,
            # Visit metadata
            "visit": visit,
            "instrument": instrument,
            # "detector": detector,
            "band": band,
            "day_obs": day_obs,
            "physical_filter": physical_filter,
            # Cross-match quality
            "sep_arcsec": sep_arcsec,
            "src_ra": m_ra,
            "src_dec": m_dec,
            # Source id
            "sourceId": int(matched_src[ID_COL]) if ID_COL in matched_src.index else np.nan,
        }

        # Add all available photometric columns
        for col in SRC_COLUMNS_AVAIL:
            if col not in (
                RA_COL,
                DEC_COL,
                ID_COL,
                "visit",
                "detector",
                "band",
                "day_obs",
                "physical_filter",
            ):
                row[col] = matched_src.get(col, np.nan)

        lc_rows.append(row)

    print(f"     matched: {n_matched}  /  {len(refs)} refs  (failed to load: {n_failed})")

    match_summary.append(
        {
            "simbad_id": simbad_id,
            "ra_deg": ra_t,
            "dec_deg": dec_t,
            "n_refs": len(refs),
            "n_matched": n_matched,
            "n_failed": n_failed,
            "status": "ok" if n_matched > 0 else "no_match",
        }
    )

print(f"\n{'='*60}")
print(f"Total LC rows collected : {len(lc_rows)}")
print(f"Stars processed         : {len(match_summary)}")

## Assemble and inspect the light-curve DataFrame


In [ ]:
df_lc = pd.DataFrame(lc_rows)
df_summary = pd.DataFrame(match_summary)

print(f"df_lc      : {len(df_lc)} rows × {len(df_lc.columns)} columns")
print(f"df_summary : {len(df_summary)} rows")

if len(df_lc) > 0:
    print("\nBand distribution:")
    print(df_lc.groupby("band")["simbad_id"].count().rename("n_measurements"))

display(df_summary)

## Save light curves

- **`all_stars_lightcurves.csv/.parquet`** — full table, all stars, all bands.
- **`per_star/`** — one CSV per star (named by `simbad_id` with special chars replaced).

> MJD timestamps will be joined in a later notebook from the visit table
> (`visitTable` / `consDb`). For now we keep `visit` and `day_obs` as integer keys.


In [ ]:
import re


def safe_name(s: str) -> str:
    """Replace characters that are invalid in filenames."""
    return re.sub(r"[^\w\-]", "_", s)


if len(df_lc) > 0:
    # ── Global table ────────────────────────────────────────────────────────
    out_all = os.path.join(DIR_DATA_OUT, "all_stars_lightcurves")
    df_lc.to_csv(out_all + ".csv", index=False)
    df_lc.to_parquet(out_all + ".parquet", index=False)
    print(f"Saved all-star LC table → {out_all}.{{csv,parquet}}")

    # ── Summary table ──────────────────────────────────────────────────
    out_sum = os.path.join(DIR_DATA_OUT, "lightcurve_match_summary")
    df_summary.to_csv(out_sum + ".csv", index=False)
    print(f"Saved match summary      → {out_sum}.csv")

    # ── Per-star tables ────────────────────────────────────────────────
    DIR_PER_STAR = os.path.join(DIR_DATA_OUT, "per_star")
    os.makedirs(DIR_PER_STAR, exist_ok=True)

    for star_id, grp in df_lc.groupby("simbad_id"):
        fname = safe_name(star_id)
        out_star = os.path.join(DIR_PER_STAR, f"{fname}_lc")
        grp.to_csv(out_star + ".csv", index=False)
        grp.to_parquet(out_star + ".parquet", index=False)

    print(f"Saved {df_lc['simbad_id'].nunique()} per-star LC files → {DIR_PER_STAR}/")
else:
    print("No LC rows to save.")

## Diagnostic plots


In [ ]:
if len(df_lc) > 0:
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.hist(df_lc["sep_arcsec"], bins=30, edgecolor="k", linewidth=0.5)
    ax.axvline(MATCH_RADIUS_ARCSEC, color="red", ls="--", label=f'search radius = {MATCH_RADIUS_ARCSEC}"')
    ax.set_xlabel("Separation (arcsec)")
    ax.set_ylabel("Number of matches")
    ax.set_title("Cross-match separation — Simbad targets vs LSST sources (all visits)")
    ax.legend()
    plt.tight_layout()
    savefig("lc_crossmatch_separation_histogram")
    plt.show()

In [ ]:
if len(df_lc) > 0:
    band_counts = df_lc.groupby(["simbad_id", "band"]).size().unstack(fill_value=0)
    fig, ax = plt.subplots(figsize=(max(6, len(band_counts) * 0.5), 4))
    band_counts.plot(kind="bar", ax=ax, width=0.8)
    ax.set_xlabel("Simbad target")
    ax.set_ylabel("Number of matched visits")
    ax.set_title("Visit count per star and band")
    ax.legend(title="band", bbox_to_anchor=(1.01, 1), loc="upper left")
    plt.xticks(rotation=45, ha="right", fontsize=7)
    plt.tight_layout()
    savefig("lc_visits_per_star_band")
    plt.show()

In [ ]:
# Plot psfFlux light curves for the first few stars, all bands
BANDS_TO_PLOT = ["r", "g", "i", "u", "z", "y"]
N_STARS_PLOT = min(6, df_lc["simbad_id"].nunique()) if len(df_lc) > 0 else 0

FLUX_COL = (
    "psfFlux" if "psfFlux" in df_lc.columns else ("calibFlux" if "calibFlux" in df_lc.columns else None)
)

if FLUX_COL and N_STARS_PLOT > 0:
    stars_to_plot = df_lc["simbad_id"].unique()[:N_STARS_PLOT]

    fig, axes = plt.subplots(
        N_STARS_PLOT,
        1,
        figsize=(10, 3 * N_STARS_PLOT),
        sharex=False,
    )
    if N_STARS_PLOT == 1:
        axes = [axes]

    BAND_COLORS = {"u": "purple", "g": "blue", "r": "green", "i": "orange", "z": "red", "y": "brown"}

    for ax, star_id in zip(axes, stars_to_plot):
        df_star = df_lc[df_lc["simbad_id"] == star_id].copy()
        df_star = df_star.sort_values(["band", "visit"])

        for band in BANDS_TO_PLOT:
            df_b = df_star[df_star["band"] == band]
            if len(df_b) == 0:
                continue
            # Use visit index as x-axis proxy (MJD will be added later)
            x = np.arange(len(df_b))
            y = df_b[FLUX_COL].values
            yerr = df_b.get(FLUX_COL + "Err", pd.Series(np.zeros(len(df_b)))).values
            color = BAND_COLORS.get(band, "gray")
            ax.errorbar(
                x, y, yerr=yerr, fmt="o", ms=3, lw=0.8, color=color, label=f"{band} ({len(df_b)} pts)"
            )

        ax.set_title(f"{star_id}", fontsize=8)
        ax.set_xlabel("Visit index (proxy — MJD to be added)")
        ax.set_ylabel(f"{FLUX_COL} [nJy]")
        ax.legend(fontsize=7, ncol=3)

    plt.suptitle(f"Light curves — {FLUX_COL} (all bands)", y=1.01)
    plt.tight_layout()
    savefig("lc_psfFlux_preview")
    plt.show()
else:
    print("No flux column available or no LC rows to plot.")

## End of notebook

The next step is to join `df_lc` with the Rubin `visitTable` (or `consDb`)
to attach precise MJD timestamps to each measurement.
This will be done in notebook `04_AddMJDFromVisitTable.ipynb`.
